In [2]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

try:
    model = load_model("Best_Gender_Model.keras")
    model_name = "Best_Gender_Model.keras"
    model_arch = "VGG16 Transfer Learning"
    model_train = "Fine-Tuning / Feature Extraction"
except OSError:
    try:
        model = load_model("Best_GENDER_Model.keras")
        model_name = "Best_GENDER_Model.keras"
        model_arch = "VGG16 Transfer Learning"
        model_train = "Fine-Tuning / Feature Extraction"
    except:
        model_name = "Unknown Model File"
        model_arch = "VGG16 / Custom CNN"
        model_train = "Transfer Learning"

class_labels = {
    0: "Female",
    1: "Male"
}

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (100, 100))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    return img

class CyberButton(tk.Canvas):
    def __init__(self, parent, text, command, width=220, height=45):
        super().__init__(parent, width=width, height=height, bg="#080c14", highlightthickness=0, cursor="hand2")
        self.command = command
        self.width = width
        self.height = height
        self.button_text = text
        
        self.bg_normal = "#0f172a"
        self.bg_hover = "#1e293b"
        self.border_normal = "#38bdf8"
        self.border_hover = "#818cf8"
        
        self.draw_button(self.bg_normal, self.border_normal, "#38bdf8")
        
        self.bind("<Enter>", self.on_enter)
        self.bind("<Leave>", self.on_leave)
        self.bind("<Button-1>", self.on_click)

    def draw_button(self, bg_color, border_color, text_color):
        self.delete("all")
        self.create_rectangle(2, 2, self.width-2, self.height-2, fill=bg_color, outline=border_color, width=1.5)
        self.create_line(2, 2, 10, 2, fill="#f43f5e", width=2)
        self.create_line(2, 2, 2, 10, fill="#f43f5e", width=2)
        self.create_line(self.width-2, self.height-2, self.width-10, self.height-2, fill="#f43f5e", width=2)
        self.create_line(self.width-2, self.height-2, self.width-2, self.height-10, fill="#f43f5e", width=2)
        self.create_text(self.width / 2, self.height / 2, text=self.button_text, fill=text_color, font=("Consolas", 10, "bold"))

    def on_enter(self, event):
        self.draw_button(self.bg_hover, self.border_hover, "#ffffff")

    def on_leave(self, event):
        self.draw_button(self.bg_normal, self.border_normal, "#38bdf8")

    def on_click(self, event):
        if self.command:
            self.command()

class GenderApp:
    def __init__(self, root):
        self.root = root
        self.root.title("NEURAL // GENDER CLASSIFICATION V1.0 [LANDSCAPE]")
        self.root.geometry("950x580")
        self.root.config(bg="#080c14")
        self.root.resizable(False, False)

        # Top Bar
        self.top_bar = tk.Frame(root, bg="#0b1329", height=32)
        self.top_bar.pack(fill="x", side="top")
        
        self.status_dot = tk.Label(self.top_bar, text="●", bg="#0b1329", fg="#10b981", font=("Consolas", 11))
        self.status_dot.pack(side="left", padx=(15, 0))
        
        self.status_text = tk.Label(self.top_bar, text="SYSTEM ONLINE // SECURE KERNEL (GENDER RECOGNITION)", bg="#0b1329", fg="#94a3b8", font=("Consolas", 8, "bold"))
        self.status_text.pack(side="left", padx=5)

        # Header Title
        self.header_frame = tk.Frame(root, bg="#080c14")
        self.header_frame.pack(fill="x", pady=(8, 4))

        self.title_label = tk.Label(
            self.header_frame, 
            text="GENDER CLASSIFICATION SYSTEM", 
            font=("Consolas", 16, "bold"), 
            bg="#080c14", 
            fg="#f8fafc"
        )
        self.title_label.pack()

        # Main Layout (Horizontal Split)
        self.content_frame = tk.Frame(root, bg="#080c14")
        self.content_frame.pack(fill="both", expand=True, padx=15, pady=8)

        # Left Column (Preview & Upload)
        self.left_panel = tk.Frame(self.content_frame, bg="#0f172a", bd=1, relief="solid", width=360)
        self.left_panel.pack(side="left", fill="both", padx=(0, 8))
        self.left_panel.config(highlightbackground="#1e293b", highlightthickness=2)
        self.left_panel.pack_propagate(False)

        self.preview_frame = tk.Frame(self.left_panel, bg="#080c14", width=180, height=180, bd=1, relief="solid")
        self.preview_frame.pack(pady=(25, 15))
        self.preview_frame.pack_propagate(False)
        self.preview_frame.config(highlightbackground="#334155", highlightthickness=1)

        self.image_label = tk.Label(self.preview_frame, bg="#080c14", text="NO INPUT IMAGE", fg="#475569", font=("Consolas", 9, "bold"))
        self.image_label.pack(fill="both", expand=True)

        self.upload_btn = CyberButton(
            self.left_panel, 
            text="UPLOAD & ANALYZE", 
            command=self.upload_image,
            width=240,
            height=40
        )
        self.upload_btn.pack(pady=(5, 20))

        # Right Column (Console & Model Info)
        self.right_panel = tk.Frame(self.content_frame, bg="#080c14")
        self.right_panel.pack(side="right", fill="both", expand=True, padx=(8, 0))

        # Console Output Box
        self.console_frame = tk.Frame(self.right_panel, bg="#05080f", bd=1, relief="solid")
        self.console_frame.pack(fill="x", pady=(0, 10))
        self.console_frame.config(highlightbackground="#1e293b", highlightthickness=1)

        self.result_label = tk.Label(
            self.console_frame, 
            text="> System waiting for target input...\n> Status: Ready for inference.", 
            font=("Consolas", 10, "bold"), 
            bg="#05080f", 
            fg="#38bdf8",
            wraplength=500,
            justify="left"
        )
        self.result_label.pack(fill="x", padx=15, pady=15)

        # Architectural Summary Box
        self.model_info_frame = tk.Frame(self.right_panel, bg="#0b1329", bd=1, relief="solid")
        self.model_info_frame.pack(fill="both", expand=True)
        self.model_info_frame.config(highlightbackground="#1e293b", highlightthickness=1)

        self.model_title_lbl = tk.Label(
            self.model_info_frame, 
            text="About Gender AI Model", 
            font=("Consolas", 10, "bold"), 
            bg="#0b1329", 
            fg="#34d399"
        )
        self.model_title_lbl.pack(anchor="w", padx=15, pady=(8, 4))

        self.model_desc_lbl = tk.Label(
            self.model_info_frame, 
            text=f"Architecture: {model_arch}\nTraining: {model_train}\nInput: 100x100 RGB (Normalized) | Classes: 2 Categories\nFramework: TensorFlow / Keras (VGG16)", 
            font=("Consolas", 9), 
            bg="#0b1329", 
            fg="#94a3b8",
            justify="left"
        )
        self.model_desc_lbl.pack(anchor="w", padx=15, pady=(0, 10))

    def upload_image(self):
        file_path = filedialog.askopenfilename(filetypes=[("Image Files", "*.png *.jpg *.jpeg *.bmp")])
        if not file_path:
            return

        try:
            pil_img = Image.open(file_path)
            pil_img = pil_img.resize((160, 160))
            self.img_tk = ImageTk.PhotoImage(pil_img)
            self.image_label.config(image=self.img_tk, text="")

            processed_img = preprocess_image(file_path)
            predictions = model.predict(processed_img)
            predicted_class = np.argmax(predictions, axis=1)[0]
            confidence = np.max(predictions) * 100

            gender_name = class_labels.get(predicted_class, "UNKNOWN")

            self.result_label.config(
                text=f"> TARGET DETECTED: {gender_name}\n> CONFIDENCE RATE: {confidence:.2f}%\n> INFERENCE STATUS: SUCCESS",
                fg="#10b981"
            )

        except Exception as e:
            messagebox.showerror("CRITICAL ERROR", f"Execution failed:\n{str(e)}")

if __name__ == "__main__":
    root = tk.Tk()
    app = GenderApp(root)
    root.mainloop()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 480ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
